# 04 — Define Summarize Composite / 04 Define Summarize Composite

**Chapter 22 — File 4 of 4 / 第22章 — 第4个文件（共4个）**

---

## Summary / 总结

This script demonstrates **example of defining a composite model for training the generator model**.

本脚本演示 **example of defining a composite model for training the generator model**。

---
## Step 1 — example of defining a composite model for training the generator model

In [ ]:
from keras.optimizers import Adam
from keras.initializers import RandomNormal
from keras.models import Model
from keras.models import Input
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Activation
from keras.layers import Concatenate
from keras.layers import Dropout
from keras.layers import BatchNormalization
from keras.layers import LeakyReLU
from keras.utils.vis_utils import plot_model

---
## Step 2 — define the discriminator model

In [ ]:
def define_discriminator(image_shape):

---
## Step 3 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 4 — source image input

In [ ]:
in_src_image = Input(shape=image_shape)

---
## Step 5 — target image input

In [ ]:
in_target_image = Input(shape=image_shape)

---
## Step 6 — concatenate images channel-wise

In [ ]:
merged = Concatenate()([in_src_image, in_target_image])

---
## Step 7 — C64

In [ ]:
d = Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(merged)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 8 — C128

In [ ]:
d = Conv2D(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 9 — C256

In [ ]:
d = Conv2D(256, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 10 — C512

In [ ]:
d = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 11 — second last output layer

In [ ]:
d = Conv2D(512, (4,4), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)

---
## Step 12 — patch output

In [ ]:
d = Conv2D(1, (4,4), padding='same', kernel_initializer=init)(d)
	patch_out = Activation('sigmoid')(d)

---
## Step 13 — define model

In [ ]:
model = Model([in_src_image, in_target_image], patch_out)

---
## Step 14 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, loss_weights=[0.5])
	return model

---
## Step 15 — define an encoder block

In [ ]:
def define_encoder_block(layer_in, n_filters, batchnorm=True):

---
## Step 16 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 17 — add downsampling layer

In [ ]:
g = Conv2D(n_filters, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(layer_in)

---
## Step 18 — conditionally add batch normalization

In [ ]:
if batchnorm:
		g = BatchNormalization()(g, training=True)

---
## Step 19 — leaky relu activation

In [ ]:
g = LeakyReLU(alpha=0.2)(g)
	return g

---
## Step 20 — define a decoder block

In [ ]:
def decoder_block(layer_in, skip_in, n_filters, dropout=True):

---
## Step 21 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 22 — add upsampling layer

In [ ]:
g = Conv2DTranspose(n_filters, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(layer_in)

---
## Step 23 — add batch normalization

In [ ]:
g = BatchNormalization()(g, training=True)

---
## Step 24 — conditionally add dropout

In [ ]:
if dropout:
		g = Dropout(0.5)(g, training=True)

---
## Step 25 — merge with skip connection

In [ ]:
g = Concatenate()([g, skip_in])

---
## Step 26 — relu activation

In [ ]:
g = Activation('relu')(g)
	return g

---
## Step 27 — define the standalone generator model

In [ ]:
def define_generator(image_shape=(256,256,3)):

---
## Step 28 — weight initialization

In [ ]:
init = RandomNormal(stddev=0.02)

---
## Step 29 — image input

In [ ]:
in_image = Input(shape=image_shape)

---
## Step 30 — encoder model: C64-C128-C256-C512-C512-C512-C512-C512

In [ ]:
e1 = define_encoder_block(in_image, 64, batchnorm=False)
	e2 = define_encoder_block(e1, 128)
	e3 = define_encoder_block(e2, 256)
	e4 = define_encoder_block(e3, 512)
	e5 = define_encoder_block(e4, 512)
	e6 = define_encoder_block(e5, 512)
	e7 = define_encoder_block(e6, 512)

---
## Step 31 — bottleneck, no batch norm and relu

In [ ]:
b = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(e7)
	b = Activation('relu')(b)

---
## Step 32 — decoder model: CD512-CD1024-CD1024-C1024-C1024-C512-C256-C128

In [ ]:
d1 = decoder_block(b, e7, 512)
	d2 = decoder_block(d1, e6, 512)
	d3 = decoder_block(d2, e5, 512)
	d4 = decoder_block(d3, e4, 512, dropout=False)
	d5 = decoder_block(d4, e3, 256, dropout=False)
	d6 = decoder_block(d5, e2, 128, dropout=False)
	d7 = decoder_block(d6, e1, 64, dropout=False)

---
## Step 33 — output

In [ ]:
g = Conv2DTranspose(3, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d7)
	out_image = Activation('tanh')(g)

---
## Step 34 — define model

In [ ]:
model = Model(in_image, out_image)
	return model

---
## Step 35 — define the combined generator and discriminator model, for updating the generator

In [ ]:
def define_gan(g_model, d_model, image_shape):

---
## Step 36 — make weights in the discriminator not trainable

In [ ]:
for layer in d_model.layers:
		if not isinstance(layer, BatchNormalization):
			layer.trainable = False

---
## Step 37 — define the source image

In [ ]:
in_src = Input(shape=image_shape)

---
## Step 38 — connect the source image to the generator input

In [ ]:
gen_out = g_model(in_src)

---
## Step 39 — connect the source input and generator output to the discriminator input

In [ ]:
dis_out = d_model([in_src, gen_out])

---
## Step 40 — src image as input, generated image and classification output

In [ ]:
model = Model(in_src, [dis_out, gen_out])

---
## Step 41 — compile model

In [ ]:
opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss=['binary_crossentropy', 'mae'], optimizer=opt, loss_weights=[1,100])
	return model

---
## Step 42 — define image shape

In [ ]:
image_shape = (256,256,3)

---
## Step 43 — define the models

In [ ]:
d_model = define_discriminator(image_shape)
g_model = define_generator(image_shape)

---
## Step 44 — define the composite model

In [ ]:
gan_model = define_gan(g_model, d_model, image_shape)

---
## Step 45 — summarize the model

In [ ]:
gan_model.summary()

---
## Step 46 — plot the model

In [ ]:
plot_model(gan_model, to_file='gan_model_plot.png', show_shapes=True, show_layer_names=True)

---
## Learning Notes / 学习笔记

- **概念**: example of defining a composite model for training the generator model 是机器学习中的常用技术。  
  *example of defining a composite model for training the generator model is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Define Summarize Composite / 04 Define Summarize Composite
# Complete Code / 完整代码
# ===============================

# example of defining a composite model for training the generator model
from keras.optimizers import Adam
from keras.initializers import RandomNormal
from keras.models import Model
from keras.models import Input
from keras.layers import Conv2D
from keras.layers import Conv2DTranspose
from keras.layers import LeakyReLU
from keras.layers import Activation
from keras.layers import Concatenate
from keras.layers import Dropout
from keras.layers import BatchNormalization
from keras.layers import LeakyReLU
from keras.utils.vis_utils import plot_model

# define the discriminator model
def define_discriminator(image_shape):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# source image input
	in_src_image = Input(shape=image_shape)
	# target image input
	in_target_image = Input(shape=image_shape)
	# concatenate images channel-wise
	merged = Concatenate()([in_src_image, in_target_image])
	# C64
	d = Conv2D(64, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(merged)
	d = LeakyReLU(alpha=0.2)(d)
	# C128
	d = Conv2D(128, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)
	# C256
	d = Conv2D(256, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)
	# C512
	d = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)
	# second last output layer
	d = Conv2D(512, (4,4), padding='same', kernel_initializer=init)(d)
	d = BatchNormalization()(d)
	d = LeakyReLU(alpha=0.2)(d)
	# patch output
	d = Conv2D(1, (4,4), padding='same', kernel_initializer=init)(d)
	patch_out = Activation('sigmoid')(d)
	# define model
	model = Model([in_src_image, in_target_image], patch_out)
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss='binary_crossentropy', optimizer=opt, loss_weights=[0.5])
	return model

# define an encoder block
def define_encoder_block(layer_in, n_filters, batchnorm=True):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# add downsampling layer
	g = Conv2D(n_filters, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(layer_in)
	# conditionally add batch normalization
	if batchnorm:
		g = BatchNormalization()(g, training=True)
	# leaky relu activation
	g = LeakyReLU(alpha=0.2)(g)
	return g

# define a decoder block
def decoder_block(layer_in, skip_in, n_filters, dropout=True):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# add upsampling layer
	g = Conv2DTranspose(n_filters, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(layer_in)
	# add batch normalization
	g = BatchNormalization()(g, training=True)
	# conditionally add dropout
	if dropout:
		g = Dropout(0.5)(g, training=True)
	# merge with skip connection
	g = Concatenate()([g, skip_in])
	# relu activation
	g = Activation('relu')(g)
	return g

# define the standalone generator model
def define_generator(image_shape=(256,256,3)):
	# weight initialization
	init = RandomNormal(stddev=0.02)
	# image input
	in_image = Input(shape=image_shape)
	# encoder model: C64-C128-C256-C512-C512-C512-C512-C512
	e1 = define_encoder_block(in_image, 64, batchnorm=False)
	e2 = define_encoder_block(e1, 128)
	e3 = define_encoder_block(e2, 256)
	e4 = define_encoder_block(e3, 512)
	e5 = define_encoder_block(e4, 512)
	e6 = define_encoder_block(e5, 512)
	e7 = define_encoder_block(e6, 512)
	# bottleneck, no batch norm and relu
	b = Conv2D(512, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(e7)
	b = Activation('relu')(b)
	# decoder model: CD512-CD1024-CD1024-C1024-C1024-C512-C256-C128
	d1 = decoder_block(b, e7, 512)
	d2 = decoder_block(d1, e6, 512)
	d3 = decoder_block(d2, e5, 512)
	d4 = decoder_block(d3, e4, 512, dropout=False)
	d5 = decoder_block(d4, e3, 256, dropout=False)
	d6 = decoder_block(d5, e2, 128, dropout=False)
	d7 = decoder_block(d6, e1, 64, dropout=False)
	# output
	g = Conv2DTranspose(3, (4,4), strides=(2,2), padding='same', kernel_initializer=init)(d7)
	out_image = Activation('tanh')(g)
	# define model
	model = Model(in_image, out_image)
	return model

# define the combined generator and discriminator model, for updating the generator
def define_gan(g_model, d_model, image_shape):
	# make weights in the discriminator not trainable
	for layer in d_model.layers:
		if not isinstance(layer, BatchNormalization):
			layer.trainable = False
	# define the source image
	in_src = Input(shape=image_shape)
	# connect the source image to the generator input
	gen_out = g_model(in_src)
	# connect the source input and generator output to the discriminator input
	dis_out = d_model([in_src, gen_out])
	# src image as input, generated image and classification output
	model = Model(in_src, [dis_out, gen_out])
	# compile model
	opt = Adam(lr=0.0002, beta_1=0.5)
	model.compile(loss=['binary_crossentropy', 'mae'], optimizer=opt, loss_weights=[1,100])
	return model

# define image shape
image_shape = (256,256,3)
# define the models
d_model = define_discriminator(image_shape)
g_model = define_generator(image_shape)
# define the composite model
gan_model = define_gan(g_model, d_model, image_shape)
# summarize the model
gan_model.summary()
# plot the model
plot_model(gan_model, to_file='gan_model_plot.png', show_shapes=True, show_layer_names=True)